# Анализ обращений в службу поддержки

## Взаимосвязи, статистические гипотезы и прогнозирование нагрузки

В этой практической работе вы пройдёте полный аналитический цикл:

1. проверите рабочую среду и исходные файлы;
2. загрузите и очистите данные;
3. объедините несколько таблиц;
4. исследуете взаимосвязи между показателями;
5. проверите статистические гипотезы;
6. подготовите временной ряд;
7. сравните базовые прогнозы и ARIMA;
8. сохраните таблицы, метрики и графики.

Данные синтетические и не содержат реальных персональных данных.

## Ожидаемый результат

После выполнения notebook в папке `outputs` должны появиться:

- `support_analysis_dataset.csv` — очищенная объединённая таблица;
- `channel_summary.csv` — показатели по каналам;
- `hypothesis_results.csv` — результаты статистических тестов;
- `daily_tickets.csv` — дневной временной ряд;
- `forecast_metrics.csv` — сравнение прогнозов;
- `forecast_values.csv` — фактические и прогнозные значения;
- папка `figures` с основными графиками.

Работайте последовательно: выполняйте ячейки сверху вниз.

## 1. Проверка рабочей среды

Notebook может быть открыт как из корня проекта, так и из папки `notebooks`. Следующая ячейка автоматически найдёт корень проекта по расположению исходных CSV-файлов.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Найти корень проекта без абсолютных путей пользователя."""
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        expected = candidate / "data" / "raw" / "support_tickets.csv"
        if expected.exists():
            return candidate

    raise FileNotFoundError(
        "Не найден корень проекта. Проверьте, что папка data/raw содержит "
        "support_tickets.csv, customers.csv и regions.csv."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Корень проекта:", PROJECT_ROOT)
print("Папка исходных данных:", RAW_DIR)
print("Папка результатов:", OUTPUT_DIR)

### Проверка файлов

Все три исходных файла должны получить статус `OK`. Если отображается `НЕ НАЙДЕН`, проверьте структуру распакованной папки.

In [ ]:
required_files = {
    "Обращения": RAW_DIR / "support_tickets.csv",
    "Клиенты": RAW_DIR / "customers.csv",
    "Регионы": RAW_DIR / "regions.csv",
}

all_files_found = True
for label, file_path in required_files.items():
    if file_path.exists():
        print(f"OK: {label}: {file_path.name}")
    else:
        all_files_found = False
        print(f"НЕ НАЙДЕН: {label}: {file_path}")

assert all_files_found, "Не все исходные файлы найдены. Исправьте структуру папок."

## 2. Импорт библиотек

Для основной работы нужны `pandas`, `numpy`, `matplotlib` и `scipy`. Библиотека `statsmodels` используется в дополнительном блоке ARIMA.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

## 3. Загрузка данных

CSV-файлы используют кодировку UTF-8 with BOM. `pandas` обычно распознаёт её автоматически; параметр `encoding="utf-8-sig"` указан явно для воспроизводимости.

In [ ]:
tickets_raw = pd.read_csv(
    RAW_DIR / "support_tickets.csv",
    encoding="utf-8-sig",
)
customers_raw = pd.read_csv(
    RAW_DIR / "customers.csv",
    encoding="utf-8-sig",
)
regions_raw = pd.read_csv(
    RAW_DIR / "regions.csv",
    encoding="utf-8-sig",
)

print("support_tickets:", tickets_raw.shape)
print("customers:", customers_raw.shape)
print("regions:", regions_raw.shape)

display(tickets_raw.head())
display(customers_raw.head())
display(regions_raw.head())

### Контрольная точка

Убедитесь, что:

- основная таблица содержит больше 1 000 строк;
- справочник клиентов содержит несколько сотен строк;
- справочник регионов содержит 8 строк;
- названия столбцов соответствуют описанию данных.

## 4. Первичная диагностика качества данных

На этом этапе данные не изменяются. Мы фиксируем исходное состояние: типы, пропуски, дубли и диапазоны числовых показателей.

In [ ]:
print("Типы данных в support_tickets:")
display(tickets_raw.dtypes.to_frame("dtype"))

print("Пропуски в support_tickets:")
display(tickets_raw.isna().sum().to_frame("missing_count"))

print("Повторные ticket_id:", tickets_raw["ticket_id"].duplicated().sum())

print("Описательные статистики:")
display(
    tickets_raw[
        ["first_response_min", "resolution_hours", "satisfaction_score"]
    ].describe()
)

### Диагностика категорий

Разный регистр и лишние пробелы создают ложные категории. Сначала посмотрим исходные значения, затем выполним нормализацию.

In [ ]:
for column in ["channel", "category", "priority"]:
    print(f"\nИсходные значения {column}:")
    display(tickets_raw[column].value_counts(dropna=False).to_frame("count"))

### Диагностика бизнес-правил

Время первого ответа не должно быть нулевым или отрицательным. Очень большое время решения не удаляется автоматически: такие значения сначала нужно исследовать как возможные выбросы или реальные сложные обращения.

In [ ]:
quality_snapshot = pd.Series({
    "rows": len(tickets_raw),
    "unique_ticket_ids": tickets_raw["ticket_id"].nunique(),
    "duplicate_ticket_rows": tickets_raw["ticket_id"].duplicated().sum(),
    "missing_satisfaction": tickets_raw["satisfaction_score"].isna().sum(),
    "missing_region": tickets_raw["region_id"].isna().sum(),
    "nonpositive_first_response": (tickets_raw["first_response_min"] <= 0).sum(),
    "resolution_over_140h": (tickets_raw["resolution_hours"] > 140).sum(),
})

display(quality_snapshot.to_frame("value"))

### Самостоятельный мини-анализ

Ответьте в новой Markdown-ячейке:

1. Какие проблемы качества критичны для расчёта корреляции?
2. Какие проблемы критичны для сравнения регионов?
3. Почему нельзя удалить все строки с любой проблемой одной командой `dropna()`?

## 5. Очистка и подготовка данных

Создаём копии исходных таблиц, чтобы сохранить необработанные данные. Решения по очистке:

- даты преобразуем в `datetime`;
- категории нормализуем через удаление пробелов и нижний регистр;
- повторные `ticket_id` удаляем, оставляя первую запись;
- значения `first_response_min <= 0` заменяем на `NaN`;
- выбросы `resolution_hours` сохраняем для дальнейшего сравнения;
- пропуски оценки не заполняем искусственно.

In [ ]:
tickets = tickets_raw.copy()
customers = customers_raw.copy()
regions = regions_raw.copy()

# Даты
tickets["created_at"] = pd.to_datetime(tickets["created_at"], errors="coerce")
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"], errors="coerce"
)

# Категории
for column in ["channel", "category", "priority"]:
    tickets[column] = (
        tickets[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

for column in ["segment", "loyalty_level"]:
    customers[column] = (
        customers[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# Бизнес-правило: время ответа должно быть положительным
tickets.loc[tickets["first_response_min"] <= 0, "first_response_min"] = np.nan

# Повторные идентификаторы обращений
tickets = tickets.drop_duplicates(subset="ticket_id", keep="first").copy()

# Логические поля
tickets["sla_breached"] = tickets["sla_breached"].astype("int8")
customers["is_premium"] = customers["is_premium"].astype("int8")

print("Строк после удаления повторных ticket_id:", len(tickets))
print("Пропуски даты created_at:", tickets["created_at"].isna().sum())
print("Неположительные first_response_min после очистки:",
      (tickets["first_response_min"] <= 0).sum())

### Проверка допустимых категорий

После нормализации ожидаются четыре канала, шесть категорий проблем и четыре уровня приоритета.

In [ ]:
expected_categories = {
    "channel": {"chat", "email", "phone", "web_form"},
    "category": {"login", "payment", "delivery", "account", "technical", "refund"},
    "priority": {"low", "normal", "high", "critical"},
}

for column, expected in expected_categories.items():
    actual = set(tickets[column].dropna().unique())
    unexpected = actual - expected
    print(f"{column}: {sorted(actual)}")
    print("  неожиданные значения:", sorted(unexpected) if unexpected else "нет")

## 6. Объединение таблиц

Основная таблица содержит много обращений на одного клиента и один регион. Поэтому объединение выполняется по схеме `many-to-one`. Параметр `validate="many_to_one"` помогает обнаружить дубли ключей в справочниках.

In [ ]:
rows_before_merge = len(tickets)

analysis_data = (
    tickets
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        regions,
        on="region_id",
        how="left",
        validate="many_to_one",
    )
)

print("Строк до merge:", rows_before_merge)
print("Строк после merge:", len(analysis_data))
print("Клиенты без совпадения:", analysis_data["segment"].isna().sum())
print("Регионы без совпадения:", analysis_data["region_name"].isna().sum())

assert len(analysis_data) == rows_before_merge, (
    "Количество строк после merge изменилось. Проверьте уникальность ключей справочников."
)

display(analysis_data.head())

### Пометка неизвестного региона

Пропуски и код `R99` не удаляются из общего анализа. Для региональных срезов они получают понятную метку `Неизвестный регион`.

In [ ]:
analysis_data["region_name_clean"] = analysis_data["region_name"].fillna(
    "Неизвестный регион"
)
analysis_data["macro_region_clean"] = analysis_data["macro_region"].fillna(
    "Неизвестно"
)

print("Распределение по регионам, включая неизвестные:")
display(
    analysis_data["region_name_clean"]
    .value_counts(dropna=False)
    .to_frame("tickets_count")
)

## 7. Базовые показатели

Перед сложными методами полезно зафиксировать основные KPI набора данных.

In [ ]:
base_metrics = pd.Series({
    "Количество уникальных обращений": analysis_data["ticket_id"].nunique(),
    "Средняя оценка клиента": analysis_data["satisfaction_score"].mean(),
    "Медианная оценка клиента": analysis_data["satisfaction_score"].median(),
    "Среднее время первого ответа, минут": analysis_data["first_response_min"].mean(),
    "Среднее время решения, часов": analysis_data["resolution_hours"].mean(),
    "Медианное время решения, часов": analysis_data["resolution_hours"].median(),
    "Доля нарушений SLA": analysis_data["sla_breached"].mean(),
})

display(base_metrics.to_frame("value"))

### Самостоятельный вывод

Сформулируйте 2–3 предложения:

- каков типичный объём времени решения;
- насколько среднее отличается от медианы;
- что это может говорить о наличии выбросов.

## 8. Анализ взаимосвязей

Корреляция показывает направление и силу согласованного изменения числовых признаков. Она не доказывает причинно-следственную связь.

Сравним коэффициенты Pearson и Spearman:

- **Pearson** оценивает линейную связь;
- **Spearman** оценивает монотонную связь по рангам и обычно менее чувствителен к выбросам.

In [ ]:
numeric_columns = [
    "first_response_min",
    "resolution_hours",
    "satisfaction_score",
    "sla_breached",
    "is_premium",
]

pearson_corr = analysis_data[numeric_columns].corr(method="pearson")
spearman_corr = analysis_data[numeric_columns].corr(method="spearman")

print("Корреляция Pearson:")
display(pearson_corr)

print("Корреляция Spearman:")
display(spearman_corr)

### Визуализация корреляционной матрицы

График строится средствами `matplotlib`, без дополнительной зависимости от `seaborn`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(pearson_corr, vmin=-1, vmax=1, cmap="coolwarm")

ax.set_xticks(range(len(numeric_columns)))
ax.set_yticks(range(len(numeric_columns)))
ax.set_xticklabels(numeric_columns, rotation=45, ha="right")
ax.set_yticklabels(numeric_columns)
ax.set_title("Корреляционная матрица Pearson")

for row in range(len(numeric_columns)):
    for col in range(len(numeric_columns)):
        ax.text(
            col,
            row,
            f"{pearson_corr.iloc[row, col]:.2f}",
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=ax, label="Коэффициент корреляции")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

### Связь времени решения и оценки клиента

Построим диаграмму рассеяния и добавим линейный тренд только по строкам без пропусков.

In [ ]:
scatter_data = analysis_data[
    ["resolution_hours", "satisfaction_score"]
].dropna()

x = scatter_data["resolution_hours"].to_numpy()
y = scatter_data["satisfaction_score"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, alpha=0.30, s=24, label="Обращения")

trend_slope, trend_intercept = np.polyfit(x, y, deg=1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = trend_slope * x_line + trend_intercept
ax.plot(x_line, y_line, linewidth=2, label="Линейный тренд")

ax.set_title("Связь времени решения и оценки клиента")
ax.set_xlabel("Время решения, часов")
ax.set_ylabel("Оценка клиента")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "resolution_vs_satisfaction.png", dpi=150, bbox_inches="tight")
plt.show()

print("Наклон линии тренда:", round(trend_slope, 4))

### Влияние выбросов на корреляцию

Сравним коэффициент на полном наборе и на данных без значений `resolution_hours > 140`. Это не означает, что выбросы обязательно нужно удалить: цель — увидеть чувствительность результата.

In [ ]:
full_corr = analysis_data[
    ["resolution_hours", "satisfaction_score"]
].corr().iloc[0, 1]

without_outliers = analysis_data.loc[
    analysis_data["resolution_hours"] <= 140,
    ["resolution_hours", "satisfaction_score"],
]
without_outliers_corr = without_outliers.corr().iloc[0, 1]

corr_comparison = pd.DataFrame({
    "dataset": ["Все данные", "Без resolution_hours > 140"],
    "pearson_corr": [full_corr, without_outliers_corr],
})

display(corr_comparison)

### Самостоятельный вывод

Ответьте:

1. Какой знак имеет связь времени решения и оценки?
2. Сильно ли меняется результат после исключения выбросов?
3. Почему фраза «долгое решение является единственной причиной низкой оценки» некорректна?

## 9. Групповой анализ и визуализация

Сравним каналы по числу обращений, оценке, времени решения и доле нарушений SLA.

In [ ]:
channel_summary = (
    analysis_data
    .groupby("channel", as_index=False)
    .agg(
        tickets_count=("ticket_id", "nunique"),
        avg_satisfaction=("satisfaction_score", "mean"),
        median_satisfaction=("satisfaction_score", "median"),
        avg_first_response_min=("first_response_min", "mean"),
        avg_resolution_hours=("resolution_hours", "mean"),
        sla_breach_rate=("sla_breached", "mean"),
    )
    .sort_values("avg_satisfaction", ascending=False)
)

display(channel_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(channel_summary["channel"], channel_summary["avg_satisfaction"])
ax.set_title("Средняя оценка клиента по каналам")
ax.set_xlabel("Канал")
ax.set_ylabel("Средняя оценка")
ax.set_ylim(0, 5)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "channel_satisfaction.png", dpi=150, bbox_inches="tight")
plt.show()

### Распределение времени решения по приоритетам

Boxplot помогает увидеть медиану, разброс и выбросы. Средние значения следует интерпретировать вместе с распределением.

In [ ]:
priority_order = ["low", "normal", "high", "critical"]
priority_values = [
    analysis_data.loc[
        analysis_data["priority"] == priority,
        "resolution_hours",
    ].dropna()
    for priority in priority_order
]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(priority_values, tick_labels=priority_order, showfliers=True)
ax.set_title("Время решения по приоритетам")
ax.set_xlabel("Приоритет")
ax.set_ylabel("Время решения, часов")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "resolution_by_priority.png", dpi=150, bbox_inches="tight")
plt.show()

### Самостоятельный шаг

Постройте аналогичную таблицу по полю `category`. Рассчитайте:

- количество обращений;
- среднее и медианное время решения;
- среднюю оценку;
- долю нарушений SLA.

Сравните категории `technical` и `refund` с остальными.

In [ ]:
# Самостоятельный шаг: измените или расширьте агрегации при необходимости.
category_summary = (
    analysis_data
    .groupby("category", as_index=False)
    .agg(
        tickets_count=("ticket_id", "nunique"),
        avg_resolution_hours=("resolution_hours", "mean"),
        median_resolution_hours=("resolution_hours", "median"),
        avg_satisfaction=("satisfaction_score", "mean"),
        sla_breach_rate=("sla_breached", "mean"),
    )
    .sort_values("avg_resolution_hours", ascending=False)
)

display(category_summary)

## 10. Тестирование статистических гипотез

Для двух независимых групп используем t-критерий Уэлча (`equal_var=False`). Он не требует считать дисперсии групп одинаковыми.

Правило интерпретации при уровне значимости `alpha = 0.05`:

- `p-value < 0.05` — есть основания отвергнуть H0;
- `p-value >= 0.05` — недостаточно оснований отвергнуть H0.

Не пишите «гипотеза доказана». Статистический тест оценивает согласованность наблюдаемых данных с H0.

In [ ]:
ALPHA = 0.05


def welch_ttest_report(
    sample_a: pd.Series,
    sample_b: pd.Series,
    label_a: str,
    label_b: str,
    metric: str,
) -> dict:
    """Выполнить t-test Уэлча и вернуть показатели для отчёта."""
    clean_a = pd.to_numeric(sample_a, errors="coerce").dropna()
    clean_b = pd.to_numeric(sample_b, errors="coerce").dropna()

    result = stats.ttest_ind(
        clean_a,
        clean_b,
        equal_var=False,
        nan_policy="omit",
    )

    return {
        "metric": metric,
        "group_a": label_a,
        "group_b": label_b,
        "n_a": len(clean_a),
        "n_b": len(clean_b),
        "mean_a": clean_a.mean(),
        "mean_b": clean_b.mean(),
        "mean_difference_a_minus_b": clean_a.mean() - clean_b.mean(),
        "t_statistic": result.statistic,
        "p_value": result.pvalue,
        "alpha": ALPHA,
        "decision": (
            "Отвергаем H0"
            if result.pvalue < ALPHA
            else "Недостаточно оснований отвергнуть H0"
        ),
    }

### Гипотеза 1. Оценка в `chat` и `email`

- **H0:** средняя оценка клиентов в каналах `chat` и `email` не отличается.
- **H1:** средняя оценка клиентов в каналах `chat` и `email` отличается.

In [ ]:
chat_scores = analysis_data.loc[
    analysis_data["channel"] == "chat",
    "satisfaction_score",
]
email_scores = analysis_data.loc[
    analysis_data["channel"] == "email",
    "satisfaction_score",
]

hypothesis_chat_email = welch_ttest_report(
    chat_scores,
    email_scores,
    label_a="chat",
    label_b="email",
    metric="satisfaction_score",
)

display(pd.DataFrame([hypothesis_chat_email]))

### Гипотеза 2. Время решения для `high` и `normal`

- **H0:** среднее время решения обращений с приоритетами `high` и `normal` не отличается.
- **H1:** среднее время решения обращений с приоритетами `high` и `normal` отличается.

In [ ]:
high_resolution = analysis_data.loc[
    analysis_data["priority"] == "high",
    "resolution_hours",
]
normal_resolution = analysis_data.loc[
    analysis_data["priority"] == "normal",
    "resolution_hours",
]

hypothesis_high_normal = welch_ttest_report(
    high_resolution,
    normal_resolution,
    label_a="high",
    label_b="normal",
    metric="resolution_hours",
)

display(pd.DataFrame([hypothesis_high_normal]))

### Гипотеза 3. Оценка при нарушенном и ненарушенном SLA

Выполните проверку по тому же шаблону.

- **H0:** средняя оценка при `sla_breached = 1` и `sla_breached = 0` не отличается.
- **H1:** средняя оценка различается.

In [ ]:
sla_breached_scores = analysis_data.loc[
    analysis_data["sla_breached"] == 1,
    "satisfaction_score",
]
sla_ok_scores = analysis_data.loc[
    analysis_data["sla_breached"] == 0,
    "satisfaction_score",
]

hypothesis_sla = welch_ttest_report(
    sla_breached_scores,
    sla_ok_scores,
    label_a="sla_breached",
    label_b="sla_ok",
    metric="satisfaction_score",
)

display(pd.DataFrame([hypothesis_sla]))

### Сводная таблица гипотез

Помимо p-value обязательно сравнивайте размеры групп и средние значения. Статистическая значимость не заменяет содержательную интерпретацию величины различия.

In [ ]:
hypothesis_results = pd.DataFrame([
    hypothesis_chat_email,
    hypothesis_high_normal,
    hypothesis_sla,
])

display(hypothesis_results)

### Самостоятельная интерпретация

Для каждой гипотезы напишите:

1. какие группы сравнивались;
2. как различаются средние;
3. чему равен p-value;
4. какое решение принято относительно H0;
5. какое ограничение есть у вывода.

## 11. Подготовка временного ряда

Для прогнозирования порядок наблюдений критичен. Данные сортируются по времени, а train/test разделяются хронологически без перемешивания.

Сначала построим число уникальных обращений по календарным дням. Дни без обращений должны присутствовать в индексе и получать значение 0.

In [ ]:
daily_tickets = (
    analysis_data
    .dropna(subset=["created_at"])
    .set_index("created_at")
    .resample("D")
    .agg(
        tickets_count=("ticket_id", "nunique"),
        avg_satisfaction=("satisfaction_score", "mean"),
        avg_resolution_hours=("resolution_hours", "mean"),
        sla_breach_rate=("sla_breached", "mean"),
    )
)

daily_tickets["tickets_count"] = daily_tickets["tickets_count"].fillna(0).astype(int)
daily_tickets["rolling_mean_7d"] = (
    daily_tickets["tickets_count"].rolling(window=7, min_periods=1).mean()
)
daily_tickets["day_of_week"] = daily_tickets.index.day_name()
daily_tickets["is_weekend"] = daily_tickets.index.dayofweek >= 5

print("Период:", daily_tickets.index.min().date(), "—", daily_tickets.index.max().date())
print("Количество календарных дней:", len(daily_tickets))
display(daily_tickets.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    daily_tickets.index,
    daily_tickets["tickets_count"],
    alpha=0.55,
    label="Фактическое число обращений",
)
ax.plot(
    daily_tickets.index,
    daily_tickets["rolling_mean_7d"],
    linewidth=2.5,
    label="Скользящее среднее за 7 дней",
)
ax.set_title("Динамика обращений в службу поддержки")
ax.set_xlabel("Дата")
ax.set_ylabel("Количество обращений")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "daily_tickets_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

### Проверка недельной сезонности и изменения нагрузки

Сравним будни и выходные, а также первую и вторую половины периода.

In [ ]:
weekday_weekend_summary = (
    daily_tickets
    .groupby("is_weekend", as_index=False)
    .agg(
        days=("tickets_count", "size"),
        avg_tickets=("tickets_count", "mean"),
        median_tickets=("tickets_count", "median"),
    )
)
weekday_weekend_summary["period_type"] = weekday_weekend_summary["is_weekend"].map({
    False: "Будни",
    True: "Выходные",
})

display(weekday_weekend_summary[["period_type", "days", "avg_tickets", "median_tickets"]])

midpoint = len(daily_tickets) // 2
first_half_mean = daily_tickets["tickets_count"].iloc[:midpoint].mean()
second_half_mean = daily_tickets["tickets_count"].iloc[midpoint:].mean()

print("Среднее в первой половине периода:", round(first_half_mean, 3))
print("Среднее во второй половине периода:", round(second_half_mean, 3))

### Самостоятельный вывод

Опишите:

- различается ли нагрузка в будни и выходные;
- заметен ли рост нагрузки во второй половине периода;
- почему эти особенности важны для выбора прогноза.

## 12. Хронологическое разделение train/test

Последние 20% дней используем как тестовый период. Будущее не должно попадать в обучающую часть.

In [ ]:
time_series = daily_tickets["tickets_count"].astype(float)

split_index = int(len(time_series) * 0.80)
train = time_series.iloc[:split_index]
test = time_series.iloc[split_index:]

print("Train:", train.index.min().date(), "—", train.index.max().date(), len(train), "дней")
print("Test:", test.index.min().date(), "—", test.index.max().date(), len(test), "дней")

assert train.index.max() < test.index.min(), "Train и test разделены некорректно."

## 13. Базовые прогнозы

Сложную модель нужно сравнивать с простыми baseline-подходами:

1. **Last value** — каждый будущий день равен последнему наблюдению train;
2. **7-day mean** — каждый будущий день равен среднему за последние 7 дней train;
3. **Seasonal naive 7** — повторяется недельный рисунок последних 7 дней.

In [ ]:
def mae(actual: pd.Series, predicted: pd.Series) -> float:
    return float(np.mean(np.abs(actual.to_numpy() - predicted.to_numpy())))


def rmse(actual: pd.Series, predicted: pd.Series) -> float:
    return float(np.sqrt(np.mean((actual.to_numpy() - predicted.to_numpy()) ** 2)))


last_value_forecast = pd.Series(train.iloc[-1], index=test.index, name="last_value")
mean_7_forecast = pd.Series(train.tail(7).mean(), index=test.index, name="mean_7")

last_week_pattern = train.tail(7).to_numpy()
seasonal_values = np.resize(last_week_pattern, len(test))
seasonal_naive_forecast = pd.Series(
    seasonal_values,
    index=test.index,
    name="seasonal_naive_7",
)

forecast_table = pd.DataFrame({
    "actual": test,
    "last_value": last_value_forecast,
    "mean_7": mean_7_forecast,
    "seasonal_naive_7": seasonal_naive_forecast,
})

display(forecast_table.head(10))

In [ ]:
baseline_metrics = []
for model_name in ["last_value", "mean_7", "seasonal_naive_7"]:
    baseline_metrics.append({
        "model": model_name,
        "MAE": mae(forecast_table["actual"], forecast_table[model_name]),
        "RMSE": rmse(forecast_table["actual"], forecast_table[model_name]),
    })

forecast_metrics = pd.DataFrame(baseline_metrics).sort_values("MAE")
display(forecast_metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index, train, label="Train")
ax.plot(test.index, test, linewidth=2.5, label="Факт test")
ax.plot(test.index, last_value_forecast, label="Last value")
ax.plot(test.index, mean_7_forecast, label="Mean 7")
ax.plot(test.index, seasonal_naive_forecast, label="Seasonal naive 7")
ax.axvline(test.index.min(), linestyle="--", alpha=0.7, label="Начало test")
ax.set_title("Сравнение базовых прогнозов")
ax.set_xlabel("Дата")
ax.set_ylabel("Количество обращений")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "baseline_forecasts.png", dpi=150, bbox_inches="tight")
plt.show()

### Самостоятельный вывод

Ответьте:

1. Какой baseline имеет минимальную MAE?
2. Совпадает ли лучший результат по MAE и RMSE?
3. Почему недельный seasonal naive может быть полезен для этого ряда?

## 14. Дополнительный блок: ARIMA

ARIMA рассматривается как расширение. Модель обучается только на train, затем прогнозирует длину test. Если `statsmodels` не установлена, ячейка не останавливает весь notebook и выводит инструкцию.

Параметры `(1, 1, 1)` используются как учебный стартовый вариант, а не как доказанно оптимальная спецификация.

In [ ]:
arima_forecast = None
arima_status = "not_run"

try:
    from statsmodels.tsa.arima.model import ARIMA

    arima_model = ARIMA(
        train,
        order=(1, 1, 1),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    arima_fit = arima_model.fit()
    arima_forecast = arima_fit.forecast(steps=len(test))
    arima_forecast.index = test.index
    arima_forecast.name = "arima_111"

    arima_metrics = pd.DataFrame([{
        "model": "arima_111",
        "MAE": mae(test, arima_forecast),
        "RMSE": rmse(test, arima_forecast),
    }])
    forecast_metrics = (
        pd.concat([forecast_metrics, arima_metrics], ignore_index=True)
        .sort_values("MAE")
        .reset_index(drop=True)
    )
    forecast_table["arima_111"] = arima_forecast
    arima_status = "success"

    display(forecast_metrics)
except ImportError:
    arima_status = "statsmodels_not_installed"
    print(
        "ARIMA пропущена: statsmodels не установлена. "
        "Установите зависимости из requirements.txt и повторите ячейку."
    )
except Exception as error:
    arima_status = f"error: {type(error).__name__}"
    print("ARIMA не построена:", error)

print("Статус ARIMA:", arima_status)

In [ ]:
if arima_forecast is not None:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train.index, train, label="Train")
    ax.plot(test.index, test, linewidth=2.5, label="Факт test")
    ax.plot(test.index, arima_forecast, label="ARIMA(1,1,1)")
    ax.axvline(test.index.min(), linestyle="--", alpha=0.7, label="Начало test")
    ax.set_title("Прогноз ARIMA и фактические значения")
    ax.set_xlabel("Дата")
    ax.set_ylabel("Количество обращений")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "arima_forecast.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("График ARIMA не построен, поскольку прогноз недоступен.")

### Интерпретация ARIMA

Сравните ARIMA с лучшим baseline. Более сложная модель полезна только тогда, когда она улучшает качество на отложенном временном периоде и остаётся содержательно объяснимой.

## 15. Сохранение результатов

Сохраняем таблицы в UTF-8 with BOM, чтобы кириллица корректно открывалась в большинстве табличных редакторов.

In [ ]:
analysis_data.to_csv(
    OUTPUT_DIR / "support_analysis_dataset.csv",
    index=False,
    encoding="utf-8-sig",
)
channel_summary.to_csv(
    OUTPUT_DIR / "channel_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
category_summary.to_csv(
    OUTPUT_DIR / "category_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
hypothesis_results.to_csv(
    OUTPUT_DIR / "hypothesis_results.csv",
    index=False,
    encoding="utf-8-sig",
)
daily_tickets.to_csv(
    OUTPUT_DIR / "daily_tickets.csv",
    encoding="utf-8-sig",
)
forecast_metrics.to_csv(
    OUTPUT_DIR / "forecast_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
forecast_table.to_csv(
    OUTPUT_DIR / "forecast_values.csv",
    encoding="utf-8-sig",
)

print("Сохранённые файлы:")
for file_path in sorted(OUTPUT_DIR.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(PROJECT_ROOT))

## 16. Финальная техническая самопроверка

Следующая ячейка проверяет ключевые признаки завершённой работы.

In [ ]:
checks = {
    "Исходные файлы найдены": all(path.exists() for path in required_files.values()),
    "Повторные ticket_id удалены": analysis_data["ticket_id"].is_unique,
    "Дата преобразована": pd.api.types.is_datetime64_any_dtype(analysis_data["created_at"]),
    "После merge строки не размножились": len(analysis_data) == rows_before_merge,
    "Корреляционная матрица построена": pearson_corr.shape[0] == len(numeric_columns),
    "Проверено не менее двух гипотез": len(hypothesis_results) >= 2,
    "Временной ряд отсортирован": daily_tickets.index.is_monotonic_increasing,
    "Train находится раньше test": train.index.max() < test.index.min(),
    "Есть метрики прогнозов": not forecast_metrics.empty,
    "Итоговый датасет сохранён": (OUTPUT_DIR / "support_analysis_dataset.csv").exists(),
}

check_table = pd.DataFrame({
    "check": list(checks.keys()),
    "passed": list(checks.values()),
})
display(check_table)

assert check_table["passed"].all(), "Некоторые контрольные проверки не пройдены."
print("Все обязательные технические проверки пройдены.")

## 17. Итоговый аналитический вывод

Заполните этот блок своими словами.

### 1. Что анализировалось

Опишите предметную область, период и основные таблицы.

### 2. Какие проблемы качества обнаружены

Укажите дубли, пропуски, грязные категории, некорректные значения и выбросы. Объясните, как они были обработаны.

### 3. Какие взаимосвязи обнаружены

Опишите знак и примерную силу наиболее заметных связей. Не подменяйте корреляцию причинностью.

### 4. Какие гипотезы проверены

Для каждой гипотезы укажите группы, средние, p-value и решение относительно H0.

### 5. Что показал временной ряд

Опишите недельную сезонность, тренд и возможные всплески.

### 6. Какой прогноз оказался лучшим

Сравните модели по MAE и RMSE. Укажите, улучшила ли ARIMA результат относительно baseline.

### 7. Ограничения анализа

Например: синтетические данные, короткий период, отсутствие внешних факторов, учебные параметры ARIMA.

### 8. Следующий аналитический шаг

Предложите одно продолжение: сегментный прогноз, моделирование сезонности, анализ регионов или подготовку BI-дашборда.

## Контрольный чек-лист

- [ ] Notebook выполнен сверху вниз без ошибок.
- [ ] Исходные данные не изменялись напрямую.
- [ ] Использованы только относительные пути.
- [ ] Даты и категориальные поля очищены.
- [ ] После `merge` проверено количество строк.
- [ ] Корреляция интерпретирована без утверждения причинности.
- [ ] Для гипотез указаны H0, H1, p-value и решение.
- [ ] Train/test разделены хронологически.
- [ ] Сложный прогноз сравнивается с baseline.
- [ ] Таблицы и графики сохранены в `outputs`.
- [ ] Итоговый вывод содержит ограничения анализа.

## 18. Интеграция с Yandex DataLens

В этом блоке подготавливаются отдельные CSV-витрины для BI-анализа. DataLens используется не вместо Python, а как следующий слой: интерактивные чарты, селекторы, дашборды и объяснение результатов пользователям.

Рабочая цепочка:

```text
очистка и статистика в Python → DataLens-витрины → чарты → дашборд → аналитический вывод
```

Подробная пошаговая инструкция находится в `materials/datalens/student_datalens_guide.md`.

In [ ]:
# Подготовка детальной витрины для Yandex DataLens
DATALENS_DIR = PROJECT_ROOT / "data" / "datalens"
DATALENS_DIR.mkdir(parents=True, exist_ok=True)

datalens_support = analysis_data.copy()
datalens_support["created_date"] = datalens_support["created_at"].dt.date.astype(str)
datalens_support["week_start"] = (
    datalens_support["created_at"]
    - pd.to_timedelta(datalens_support["created_at"].dt.dayofweek, unit="D")
).dt.normalize().dt.date.astype(str)
datalens_support["month_start"] = datalens_support["created_at"].dt.to_period("M").dt.start_time.dt.date.astype(str)
datalens_support["hour"] = datalens_support["created_at"].dt.hour
datalens_support["weekday_number"] = datalens_support["created_at"].dt.dayofweek + 1
weekday_ru = {0:"Понедельник",1:"Вторник",2:"Среда",3:"Четверг",4:"Пятница",5:"Суббота",6:"Воскресенье"}
datalens_support["weekday_name"] = datalens_support["created_at"].dt.dayofweek.map(weekday_ru)
datalens_support["is_weekend"] = (datalens_support["created_at"].dt.dayofweek >= 5).astype(int)
mid_date = datalens_support["created_at"].min() + (
    datalens_support["created_at"].max() - datalens_support["created_at"].min()
) / 2
datalens_support["period_part"] = np.where(
    datalens_support["created_at"] < mid_date, "Первая половина", "Вторая половина"
)
datalens_support["record_count"] = 1
datalens_support["sla_status"] = np.where(
    datalens_support["sla_breached"].eq(1), "SLA нарушен", "SLA выполнен"
)
datalens_support["satisfaction_band"] = pd.cut(
    datalens_support["satisfaction_score"], [-np.inf, 3, 4.5, np.inf],
    labels=["Низкая", "Средняя", "Высокая"], right=False,
).astype("string")
datalens_support["first_response_bucket"] = pd.cut(
    datalens_support["first_response_min"], [-np.inf, 5, 15, 30, 60, np.inf],
    labels=["До 5 мин", "5–15 мин", "15–30 мин", "30–60 мин", "Более 60 мин"], right=False,
).astype("string")
datalens_support["resolution_bucket"] = pd.cut(
    datalens_support["resolution_hours"], [-np.inf, 8, 24, 48, 72, np.inf],
    labels=["До 8 ч", "8–24 ч", "24–48 ч", "48–72 ч", "Более 72 ч"], right=False,
).astype("string")
datalens_support["channel_hypothesis_group"] = np.where(
    datalens_support["channel"].isin(["chat", "email"]), datalens_support["channel"], "other"
)
datalens_support["priority_hypothesis_group"] = np.where(
    datalens_support["priority"].isin(["high", "normal"]), datalens_support["priority"], "other"
)
datalens_support["sla_hypothesis_group"] = np.where(
    datalens_support["sla_breached"].eq(1), "sla_breached", "sla_ok"
)
datalens_support["created_at"] = datalens_support["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
if pd.api.types.is_datetime64_any_dtype(datalens_support["registration_date"]):
    datalens_support["registration_date"] = datalens_support["registration_date"].dt.date.astype(str)

datalens_support.to_csv(
    DATALENS_DIR / "support_service_datalens.csv", index=False, encoding="utf-8-sig"
)
print("Детальная витрина:", datalens_support.shape)


In [ ]:
# Дневная витрина, статистические гипотезы и метрики прогнозов
import math

datalens_daily = daily_tickets.reset_index().rename(columns={"created_at": "created_date"})
datalens_daily["created_date"] = pd.to_datetime(datalens_daily["created_date"])
forecast_export = forecast_table.reset_index().rename(columns={"created_at": "created_date", "index": "created_date"})
forecast_export["created_date"] = pd.to_datetime(forecast_export["created_date"])
datalens_daily = datalens_daily.merge(forecast_export, on="created_date", how="left")
datalens_daily["dataset_part"] = np.where(datalens_daily["actual"].notna(), "test", "train")
for model_name in ["last_value", "mean_7", "seasonal_naive_7", "arima_111"]:
    if model_name in datalens_daily:
        datalens_daily[f"abs_error_{model_name}"] = np.where(
            datalens_daily["actual"].notna() & datalens_daily[model_name].notna(),
            (datalens_daily["actual"] - datalens_daily[model_name]).abs(),
            np.nan,
        )
datalens_daily["week_start"] = (
    datalens_daily["created_date"]
    - pd.to_timedelta(datalens_daily["created_date"].dt.dayofweek, unit="D")
).dt.normalize().dt.date.astype(str)
datalens_daily["weekday_number"] = datalens_daily["created_date"].dt.dayofweek + 1
datalens_daily["weekday_name"] = datalens_daily["created_date"].dt.dayofweek.map(weekday_ru)
datalens_daily["created_date"] = datalens_daily["created_date"].dt.date.astype(str)
datalens_daily.to_csv(
    DATALENS_DIR / "daily_forecast_datalens.csv", index=False, encoding="utf-8-sig"
)

def cohens_d_for_export(x, y):
    x, y = pd.Series(x).dropna().astype(float), pd.Series(y).dropna().astype(float)
    pooled = math.sqrt(((len(x)-1)*x.var(ddof=1) + (len(y)-1)*y.var(ddof=1)) / (len(x)+len(y)-2))
    return (x.mean() - y.mean()) / pooled

hypothesis_specs = [
    ("H1", "Средняя оценка: chat против email", "Отличается ли средняя оценка клиента между chat и email?", "satisfaction_score", "channel", ["chat", "email"], ["chat", "email"]),
    ("H2", "Время решения: high против normal", "Отличается ли среднее время решения между high и normal?", "resolution_hours", "priority", ["high", "normal"], ["high", "normal"]),
    ("H3", "Оценка: SLA нарушен против выполнен", "Отличается ли оценка при нарушенном и выполненном SLA?", "satisfaction_score", "sla_breached", [1, 0], ["sla_breached", "sla_ok"]),
]
hypothesis_rows = []
for index, (hid, title, question, metric, group_field, group_values, labels) in enumerate(hypothesis_specs):
    samples = [analysis_data.loc[analysis_data[group_field] == value, metric].dropna() for value in group_values]
    effect = cohens_d_for_export(samples[0], samples[1])
    result_row = hypothesis_results.iloc[index]
    magnitude = "Большой" if abs(effect) >= 0.8 else "Средний" if abs(effect) >= 0.5 else "Малый" if abs(effect) >= 0.2 else "Очень малый"
    for order, (label, values) in enumerate(zip(labels, samples), start=1):
        hypothesis_rows.append({
            "hypothesis_id": hid,
            "hypothesis_title": title,
            "question": question,
            "metric": metric,
            "group_name": label,
            "sample_size": len(values),
            "mean_value": values.mean(),
            "median_value": values.median(),
            "std_value": values.std(ddof=1),
            "min_value": values.min(),
            "max_value": values.max(),
            "p_value": result_row["p_value"],
            "alpha": result_row["alpha"],
            "statistically_significant": "Да" if result_row["p_value"] < result_row["alpha"] else "Нет",
            "decision": result_row["decision"],
            "mean_difference_a_minus_b": result_row["mean_difference_a_minus_b"],
            "cohens_d_a_minus_b": effect,
            "effect_magnitude": magnitude,
            "group_order": order,
        })

hypothesis_summary_datalens = pd.DataFrame(hypothesis_rows)
hypothesis_summary_datalens.to_csv(
    DATALENS_DIR / "hypothesis_summary_datalens.csv", index=False, encoding="utf-8-sig"
)

forecast_metrics_datalens = forecast_metrics.melt(
    id_vars="model", var_name="metric", value_name="value"
)
forecast_metrics_datalens.to_csv(
    DATALENS_DIR / "forecast_metrics_datalens.csv", index=False, encoding="utf-8-sig"
)

print("Дневная витрина:", datalens_daily.shape)
print("Сводка гипотез:", hypothesis_summary_datalens.shape)
print("Метрики прогнозов:", forecast_metrics_datalens.shape)


### Гипотезы через чарты

DataLens помогает увидеть различие и сформулировать вопрос, но график не заменяет статистический тест. Для каждой гипотезы на дашборде покажите:

1. средние значения групп;
2. размер каждой группы;
3. p-value, рассчитанный в Python;
4. практическую величину различия;
5. ограничение причинной интерпретации.

При изменении селекторов статистический тест для нового среза нужно пересчитать в Python.

In [ ]:
datalens_checks = {
    "Детальная витрина создана": (DATALENS_DIR / "support_service_datalens.csv").exists(),
    "Дневная витрина создана": (DATALENS_DIR / "daily_forecast_datalens.csv").exists(),
    "Сводка гипотез создана": (DATALENS_DIR / "hypothesis_summary_datalens.csv").exists(),
    "Метрики прогнозов созданы": (DATALENS_DIR / "forecast_metrics_datalens.csv").exists(),
    "Одна строка детальной витрины соответствует обращению": datalens_support["ticket_id"].is_unique,
}

display(pd.DataFrame({"check": datalens_checks.keys(), "passed": datalens_checks.values()}))
assert all(datalens_checks.values()), "Не все DataLens-витрины готовы."
print("DataLens-витрины готовы к загрузке.")

### Что собрать в DataLens

- операционный обзор с четырьмя KPI;
- динамику обращений;
- сравнение каналов и категорий;
- точечный чарт времени решения и оценки;
- лабораторию трёх гипотез;
- сравнение прогнозов по MAE и RMSE;
- селекторы периода, региона, канала и категории.

Используйте чек-лист `materials/datalens/datalens_qa_checklist.md`.